In [1]:
import pandas as pd
import numpy as np

historical_df = pd.read_csv(
    "../data/interim/historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"]
)

historical_df["match_id"] = historical_df.index

historical_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_id
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,0
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,2
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,3
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,4


In [2]:
home_matches = pd.DataFrame({
    "match_id": historical_df["match_id"],
    "date": historical_df["date"],
    "team": historical_df["home_team"],
    "opponent": historical_df["away_team"],
    "goals_for": historical_df["home_score"],
    "goals_against": historical_df["away_score"],
    "is_home": 1
})

In [3]:
away_matches = pd.DataFrame({
    "match_id": historical_df["match_id"],
    "date": historical_df["date"],
    "team": historical_df["away_team"],
    "opponent": historical_df["home_team"],
    "goals_for": historical_df["away_score"],
    "goals_against": historical_df["home_score"],
    "is_home": 0
})

In [4]:
home_matches["points"] = np.where(
    home_matches["goals_for"] > home_matches["goals_against"],
    3,
    np.where(
        home_matches["goals_for"] == home_matches["goals_against"],
        1,
        0
    )
)

In [5]:
away_matches["points"] = np.where(
    away_matches["goals_for"] > away_matches["goals_against"],
    3,
    np.where(
        away_matches["goals_for"] == away_matches["goals_against"],
        1,
        0
    )
)

In [6]:
team_matches_df = pd.concat(
    [home_matches, away_matches],
    ignore_index=True
)

In [7]:
team_matches_df = team_matches_df.sort_values(
    by="date"
).reset_index(drop=True)

In [8]:
team_matches_df.shape

(98808, 8)

In [9]:
team_matches_df.head(10)

,match_id,date,team,opponent,goals_for,goals_against,is_home,points
0,0,1872-11-30,Scotland,England,0,0,1,1
1,0,1872-11-30,England,Scotland,0,0,0,1
2,1,1873-03-08,Scotland,England,2,4,0,0
3,1,1873-03-08,England,Scotland,4,2,1,3
4,2,1874-03-07,Scotland,England,2,1,1,3
5,2,1874-03-07,England,Scotland,1,2,0,0
6,3,1875-03-06,England,Scotland,2,2,1,1
7,3,1875-03-06,Scotland,England,2,2,0,1
8,4,1876-03-04,Scotland,England,3,0,1,3
9,4,1876-03-04,England,Scotland,0,3,0,0


In [11]:
team_matches_df = team_matches_df.sort_values(
    ["team", "date"]
)

In [12]:
team_matches_df["form_points_5"] = (
    team_matches_df
    .groupby("team")["points"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(
             window=5,
             min_periods=1
         )
         .sum()
    )
)

In [13]:
team_matches_df["avg_goals_for_5"] = (
    team_matches_df
    .groupby("team")["goals_for"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(
             window=5,
             min_periods=1
         )
         .mean()
    )
)

In [14]:
team_matches_df["avg_goals_against_5"] = (
    team_matches_df
    .groupby("team")["goals_against"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(
             window=5,
             min_periods=1
         )
         .mean()
    )
)

In [15]:
team_matches_df[
    team_matches_df["team"] == "Argentina"
].tail(10)

,match_id,date,team,opponent,goals_for,goals_against,is_home,points,form_points_5,avg_goals_for_5,avg_goals_against_5
96840,48439,2025-06-10,Argentina,Colombia,1,1,1,1,12.0,1.6,0.6
97062,48534,2025-09-04,Argentina,Venezuela,3,0,1,3,13.0,1.6,0.4
97301,48647,2025-09-09,Argentina,Ecuador,0,1,0,0,13.0,2.0,0.4
97484,48731,2025-10-10,Argentina,Venezuela,1,0,1,3,10.0,1.8,0.6
97664,48817,2025-10-14,Argentina,Puerto Rico,6,0,0,3,10.0,1.2,0.4
97811,48894,2025-11-14,Argentina,Angola,2,0,0,3,10.0,2.2,0.4
98317,49147,2026-03-27,Argentina,Mauritania,2,1,1,3,12.0,2.4,0.2
98499,49217,2026-03-31,Argentina,Zambia,5,0,1,3,12.0,2.2,0.4
98684,49341,2026-06-06,Argentina,Honduras,2,0,1,3,15.0,3.2,0.2
98777,49379,2026-06-09,Argentina,Iceland,3,0,1,3,15.0,3.4,0.2


In [16]:
team_matches_df["goal_diff"] = (
    team_matches_df["goals_for"]
    - team_matches_df["goals_against"]
)

In [17]:
team_matches_df["avg_goal_diff_5"] = (
    team_matches_df
    .groupby("team")["goal_diff"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(
             window=5,
             min_periods=1
         )
         .mean()
    )
)

In [18]:
print(team_matches_df["team"].nunique())

336


In [19]:
sorted(team_matches_df["team"].unique())[:20]

['Abkhazia',
 'Afghanistan',
 'Albania',
 'Alderney',
 'Algeria',
 'Ambazonia',
 'American Samoa',
 'Andalusia',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Arameans Suryoye',
 'Argentina',
 'Armenia',
 'Artsakh',
 'Aruba',
 'Asturias',
 'Australia',
 'Austria']

In [20]:
team_matches_df.columns.tolist()

['match_id',
 'date',
 'team',
 'opponent',
 'goals_for',
 'goals_against',
 'is_home',
 'points',
 'form_points_5',
 'avg_goals_for_5',
 'avg_goals_against_5',
 'goal_diff',
 'avg_goal_diff_5']

In [21]:
team_matches_df.shape

(98808, 13)

In [22]:
team_matches_df.to_csv(
    "../data/interim/team_matches.csv",
    index=False
)